# Phase-4.2 — Evaluation-Improvement Pass (matched coverage, ablations, rigor)

All numbers are recomputed **here** from the frozen Phase-4 prediction artifacts
(`artifacts/phase4/**`, read-only) by re-running `scripts/eval_matched_coverage.py` in the
background kernel session. Nothing is refit on the test; matched-coverage thresholds are
analysis-time descriptives for method comparison, not new pre-registered operating points.


In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
BASE = Path.cwd()
if not (BASE / 'scripts' / 'eval_matched_coverage.py').exists():
    BASE = BASE.parent            # kernel cwd is notebooks/
sys.path.insert(0, str(BASE / 'scripts'))
from eval_matched_coverage import selective_risk, stable_topk, bootstrap_risk_diff2, wilson

OUT = BASE / 'artifacts/phase4_2/analysis'; OUT.mkdir(parents=True, exist_ok=True)
res = __import__('runpy').run_path(str(BASE / 'scripts' / 'eval_matched_coverage.py'))
res['main']()   # recompute and rewrite analysis artifacts from frozen CSVs
summary = json.loads((BASE / 'artifacts/phase4_2/analysis/summary.json').read_text())
print('strategy rows:', len(summary['matched_coverage']))
print('n_finqa_test:', summary['n_finqa_test'])

wrote /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4_2/analysis
{
 "theta_0.80": {
  "theta": 0.409,
  "cov": 0.7855274629468177,
  "risk": 0.23751387347391784
 },
 "theta_0.90": {
  "theta": 0.2211,
  "cov": 0.925893635571055,
  "risk": 0.300376647834275
 }
}
{
 "cov": 0.8,
 "frozen_pi_vs_greedy_confidence": {
  "mean": -0.07026644856415849,
  "ci95": [
   -0.08942202835332602,
   -0.05016357688113415
  ],
  "p_better_than_A": 1.0
 },
 "agree_only_vs_greedy_confidence": {
  "mean": -0.07358705925118139,
  "ci95": [
   -0.09269356597600864,
   -0.05452562704471109
  ],
  "p_better_than_A": 1.0
 },
 "frozen_pi_vs_agree_only": {
  "mean": 0.0033206106870229007,
  "ci95": [
   -0.0032715376226826187,
   0.009814612868047967
  ],
  "p_better_than_A": 0.09733333333333333
 }
}
{
 "greedy_confidence::greedy_exec_correct": 0.5779186155601002,
 "c0_print_confidence::final_answer_correct": 0.80520075166981,
 "self_consistency_print::final_answer_correct": 0.7880980

## 1. Matched-coverage selective risk (FinQA, exec channel)

In [2]:
mc = pd.DataFrame(summary['matched_coverage'])
exec_strats = ['greedy_confidence','best_of_n_exec','exec_verified_agree_only',
               'exec_verified_agree_x_exec','exec_verified_frozen_pi']
t = mc[mc.strategy.isin(exec_strats)].pivot(index='coverage', columns='strategy', values='selective_acc')
print('Selective ACCURACY (higher is better) --- greedy full = 0.6574 \n')
print(t.round(3))
t2 = mc[mc.strategy.isin(exec_strats)].pivot(index='coverage', columns='strategy', values='selective_risk')
print('\nSelective RISK (lower is better) --- greedy full risk = 0.3426 \n')
print(t2.round(3))

Selective ACCURACY (higher is better) --- greedy full = 0.6574 

strategy  best_of_n_exec  exec_verified_agree_only  \
coverage                                             
0.50               0.660                     0.818   
0.70               0.638                     0.794   
0.80               0.654                     0.758   
0.90               0.659                     0.715   
0.95               0.663                     0.687   

strategy  exec_verified_agree_x_exec  exec_verified_frozen_pi  \
coverage                                                        
0.50                           0.818                    0.818   
0.70                           0.794                    0.794   
0.80                           0.758                    0.754   
0.90                           0.714                    0.710   
0.95                           0.686                    0.689   

strategy  greedy_confidence  
coverage                     
0.50                  0.674  
0.70      

## 2. Paired significance at matched coverage 0.80 (bootstrap B=3000)

In [3]:
print(json.dumps(summary['matched_cov_0.80_bootstrap'], indent=1))

{
 "cov": 0.8,
 "frozen_pi_vs_greedy_confidence": {
  "mean": -0.07026644856415849,
  "ci95": [
   -0.08942202835332602,
   -0.05016357688113415
  ],
  "p_better_than_A": 1.0
 },
 "agree_only_vs_greedy_confidence": {
  "mean": -0.07358705925118139,
  "ci95": [
   -0.09269356597600864,
   -0.05452562704471109
  ],
  "p_better_than_A": 1.0
 },
 "frozen_pi_vs_agree_only": {
  "mean": 0.0033206106870229007,
  "ci95": [
   -0.0032715376226826187,
   0.009814612868047967
  ],
  "p_better_than_A": 0.09733333333333333
 }
}


## 3. Ranking quality (ROC-AUC) + ablations

In [4]:
auc = pd.Series(summary['score_roc_auc']).sort_values(ascending=False)
print(auc.round(3).to_string())
print('\nAblation verdict: agreement is necessary+sufficient; executability ~0.51 (chance).')

c0_print_confidence::final_answer_correct        0.805
self_consistency_print::final_answer_correct     0.788
exec_verified_frozen_pi::execution_correct       0.763
exec_verified_agree_only::execution_correct      0.758
exec_verified_agree_x_exec::execution_correct    0.757
greedy_confidence::greedy_exec_correct           0.578
best_of_n_exec::execution_correct                0.535
exec_verified_executable_rate                    0.510

Ablation verdict: agreement is necessary+sufficient; executability ~0.51 (chance).


## 4. Error-taxonomy reconciliation (partitions sum to n=1147)

In [5]:
import pprint
print(json.dumps(summary['error_reconciliation'], indent=1))

{
 "greedy_full_partition": {
  "parse_fail": 0,
  "invalid_program": 3,
  "exec_and_print_correct": 399,
  "format_mismatch_exec_ok": 355,
  "exec_wrong_and_print_wrong": 388,
  "exec_wrong_print_correct": 2
 },
 "exec_wrong_overall": 393,
 "greedy_exec_channel": {
  "parse_fail": 0,
  "invalid_program": 3,
  "executed_wrong_or_invalid": 393,
  "exec_correct": 754,
  "n": 1147
 },
 "greedy_print_channel": {
  "print_correct": 401,
  "print_wrong": 746,
  "n": 1147
 },
 "exec_verified_exec_channel": {
  "invalid_program": 0,
  "executed_wrong": 395,
  "exec_correct": 752,
  "answer_program_mismatch": 0,
  "n": 1147
 },
 "rates": {
  "greedy_exec_acc": 0.6573670444638187,
  "greedy_print_acc": 0.34960767218831734,
  "greedy_format_mismatch_rate": 0.3095030514385353,
  "greedy_print_only_correct_note": "printed string matches gold while executed value differs (n=2)",
  "evg_exec_acc": 0.6556233653007847,
  "evg_invalid_rate": 0.0
 }
}


## 5. FiQA regression — matched-coverage selective MAE

In [6]:
fq = JSON = None
import json as _j
fq = _j.loads((BASE / 'artifacts/phase4_2/analysis/fiqa_extra.json').read_text())
fd = pd.DataFrame(fq['matched_coverage'])
print(fd[fd.coverage.isin([0.5,0.8])].round(4).to_string(index=False))
print('\nAUC(correct@0.10 vs score):', {k: round(v,3) for k,v in fq['auc_correct_vs_score'].items()})

               strategy  coverage  n_answered  selective_mae  selective_acc0.10  full_mae  mae_reduction_vs_greedy_full
      greedy_confidence       0.5         117         0.1009             0.5812    0.1301                        0.2243
      greedy_confidence       0.8         187         0.1176             0.5775    0.1301                        0.0962
      median_confidence       0.5         117         0.1063             0.5470    0.1306                        0.1832
      median_confidence       0.8         187         0.1134             0.5722    0.1306                        0.1281
median_agreement_negstd       0.5         117         0.0919             0.6581    0.1306                        0.2937
median_agreement_negstd       0.8         187         0.1094             0.5936    0.1306                        0.1595

AUC(correct@0.10 vs score): {'greedy_confidence': 0.521, 'median_confidence': 0.535, 'median_agreement_negstd': 0.597}


## 6. Risk-coverage plots

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plots = BASE / 'artifacts/phase4_2/plots'; plots.mkdir(parents=True, exist_ok=True)

def plot(metric, title, fname, lower_better=True):
    fig, ax = plt.subplots(figsize=(6.5,4.5))
    for s in exec_strats:
        sub = mc[mc.strategy==s].sort_values('coverage')
        ax.plot(sub.coverage, sub[metric], marker='o', ms=4, label=s.replace('_',' '))
    ax.axhline(0.3426, ls='--', color='grey', lw=1, label='greedy full risk')
    ax.set_xlabel('coverage'); ax.set_ylabel(title); ax.set_title(title); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(plots/fname, dpi=130); plt.close(fig)
    print('saved', plots/fname)

plot('selective_risk', 'Selective risk (FinQA exec) — lower is better', 'fina_risk_coverage.png')
plot('selective_acc', 'Selective accuracy (FinQA exec) — higher is better', 'fina_acc_coverage.png')

fd2 = fd[fd.strategy.isin(['greedy_confidence','median_confidence','median_agreement_negstd'])]
fig, ax = plt.subplots(figsize=(6.5,4.5))
for s in ['greedy_confidence','median_confidence','median_agreement_negstd']:
    sub = fd2[fd2.strategy==s].sort_values('coverage')
    ax.plot(sub.coverage, sub.selective_mae, marker='o', ms=4, label=s.replace('_',' '))
ax.axhline(0.1301, ls='--', color='grey', lw=1, label='greedy full MAE')
ax.set_xlabel('coverage'); ax.set_ylabel('selective MAE'); ax.set_title('FiQA selective MAE — lower is better'); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(plots/'fiqa_mae_coverage.png', dpi=130); plt.close(fig)
print('saved', plots/'fiqa_mae_coverage.png')

saved /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4_2/plots/fina_risk_coverage.png
saved /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4_2/plots/fina_acc_coverage.png
saved /home/heet18/Downloads/Research Papers/trust_fin_extra_phase/artifacts/phase4_2/plots/fiqa_mae_coverage.png


## 7. Interpretation

- At matched coverage, agreement-exec-verified abstention beats **confidence-based**
  abstention (paired bootstrap CI for the risk difference excludes zero, P=1.0 on FinQA) and
  best-of-n.
- Agreement is the necessary feature (AUC 0.758 vs 0.578 confidence); executability adds
  nothing; Platt-full p̂ adds nothing beyond agreement (P=0.097), but gives the pre-registered
  threshold its probability meaning.
- Error taxonomy reconciled: the only channel EVG leaves open is the 355-record format
  (printed-string) channel.
- FiQA: agreement ranks best (AUC 0.597, selective-MAE best at every coverage) though effects
  are small (n=234) — descriptive only.
- C1 remains rejected: no accuracy gain anywhere; EVG's value is purely selective reliability.

Full prose: `docs/exp_4(c).md`. Frozen record untouched: `artifacts/phase4/`. New analysis
landed in `artifacts/phase4_2/`.